In [1]:
import numpy as np
import pandas as pd
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

from sklearn.compose import ColumnTransformer
from imblearn.pipeline import Pipeline as ImbPipeline

from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, roc_auc_score
from sklearn import set_config

# For experiment analysis
import itertools
import statistics
import matplotlib.pyplot as plt

import exp as exptbl

In [2]:
categorical_features = ["workclass", "marital-status", "occupation", "relationship", "race", "native-country"]
ordinal_features = ['age', 'hours-per-week', "education-num"]
binary_features = ['sex']

In [3]:
# Load and begin prepping strip search data
df = pd.read_csv('./datasets/torontostripsearch.csv', delimiter=',')

df['IsYouth'] = np.where(df['Youth_at_arrest__under_18_years'] == "Not a youth", 0, 1)
df = df.rename(columns={"Arrest_Month" : "Arrest_Quarter"})
df.drop(columns=df.columns.intersection(['SearchReason_CauseInjury', 'SearchReason_AssistEscape', 'SearchReason_PossessWeapons', 'Arrest_Year',
                                         'SearchReason_PossessEvidence', 'Youth_at_arrest__under_18_years', "_defensive_or_escape_risk",
                                         'ObjectId', 'EventID', 'ArrestID', 'PersonID', 'Booked', 'ItemsFound']), axis=1, inplace=True)

df.replace({'Perceived_Race': {np.nan: 'Unknown or Legacy'}}, inplace=True)

df = df.drop(df[df['Sex'] == 'U'].index)

df.dropna(how="any", inplace=True)
categorical_features = ['Perceived_Race', 'Sex', 'Occurrence_Category', 'ArrestLocDiv']
ordinal_features = ['Age_group__at_arrest_', 'Arrest_Quarter']
binary_features = ['IsYouth','Actions_at_arrest___Concealed_i', 'Actions_at_arrest___Combative__',
                  'Actions_at_arrest___Resisted__d', 'Actions_at_arrest___Mental_inst',
                  'Actions_at_arrest___Assaulted_o', 'Actions_at_arrest___Cooperative']


X = df.drop('StripSearch', axis=1)
X = X.drop('IsYouth', axis=1)
y = df['StripSearch']

# Map the age groups to the specified values
custom_mapping = {
    'Aged 17 years and under': 0,
    'Aged 17 years and younger': 0,
    'Aged 18 to 24 years': 1,
    'Aged 25 to 34 years': 2,
    'Aged 35 to 44 years': 3,
    'Aged 45 to 54 years': 4,
    'Aged 55 to 64 years': 5,
    'Aged 65 and older': 6,
    'Aged 65 years and older': 6
}

quarter_mapping = {
"Jan-Mar" : 0,
"Apr-June" : 1,
"July-Sept" : 3,
"Oct-Dec" : 4
}

X['Age_group__at_arrest_'] = X['Age_group__at_arrest_'].map(custom_mapping)
X['Arrest_Quarter'] = X['Arrest_Quarter'].map(quarter_mapping)


rev_custom_mapping = {
    0 : 'Aged 17 years and younger',
    1 : 'Aged 18 to 24 years'      ,
    2 : 'Aged 25 to 34 years'      ,
    3 : 'Aged 35 to 44 years'      ,
    4 : 'Aged 45 to 54 years'      ,
    5 : 'Aged 55 to 64 years'      ,
    6 : 'Aged 65 and older'        
}

rev_quarter_mapping = {
    0 : "Jan-Mar"   ,
    1 : "Apr-June"  ,
    3 : "July-Sept" ,
    4 : "Oct-Dec"   
}

custom_encoders = {
    'Age_group__at_arrest_': rev_custom_mapping,
    'Arrest_Quarter': rev_quarter_mapping
}



In [4]:
# Create train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


# Apply label encoding for categorical features
encoders = {}
for feature in categorical_features + ordinal_features:
    le = LabelEncoder()
    X_train[feature] = le.fit_transform(X_train[feature])
    X_test[feature] = le.transform(X_test[feature])
    print(feature)
    for index, label in enumerate(le.classes_):
      print(f"Label {index}: {label}")
    encoders[feature] = le

      

Perceived_Race
Label 0: Black
Label 1: East/Southeast Asian
Label 2: Indigenous
Label 3: Latino
Label 4: Middle-Eastern
Label 5: South Asian
Label 6: Unknown or Legacy
Label 7: White
Sex
Label 0: F
Label 1: M
Occurrence_Category
Label 0: Assault
Label 1: Assault & Other crimes against persons
Label 2: Break & Enter
Label 3: Break and Enter
Label 4: Crimes against Children
Label 5: Drug Related
Label 6: FTA/FTC, Compliance Check & Parollee
Label 7: FTA/FTC/Compliance Check/Parollee
Label 8: Fraud
Label 9: Harassment & Threatening
Label 10: Harassment/Threatening
Label 11: Homicide
Label 12: Impaired
Label 13: LLA
Label 14: Mental Health
Label 15: Mischief
Label 16: Mischief & Fraud
Label 17: Other Offence
Label 18: Other Statute
Label 19: Other Statute & Other Incident Type
Label 20: Police Category - Administrative
Label 21: Police Category - Incident
Label 22: Robbery & Theft
Label 23: Robbery/Theft
Label 24: Sexual Related Crime
Label 25: Sexual Related Crimes & Crimes Against Childr

In [ ]:
# Train xgboost classifier from sklearn on it
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ordinal_features)], remainder='passthrough')

xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.2, max_depth=5, gamma = 0, min_child_weight=5, subsample=1,
                          eval_metric = "logloss", random_state=42)

xgb_model = ImbPipeline([
    ('preprocessor', preprocessor),
    ('classifier', xgb_model)])

xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)

print(f'XGBoost Test Accuracy: {accuracy_score(y_test, y_pred_xgb):.3f}')
print(f'XGBoost Test F1: {f1_score(y_test, y_pred_xgb):.3f}')


XGBoost Test Accuracy: 0.900
XGBoost Test F1: 0.474


In [6]:
y_test.shape

(13016,)

In [7]:
y_pred_xgb.shape

(13016,)

In [8]:
def label_false_positives(X_test, y_test, y_pred):
    fps = np.zeros(y_pred.shape[0])
    y_truth = y_test.copy().reset_index(drop=True)
    test_copy = X_test.copy() 
    for i in test_copy.reset_index(drop=True).itertuples():
        if y_pred[i[0]] == 1 and y_truth[i[0]] == 0:
            fps[i[0]] = 1
    test_copy['targetcol'] = np.round(fps, 2)
    return test_copy
    

In [9]:
def label_false_negatives(X_test, y_test, y_pred):
    fps = np.zeros(y_pred.shape[0])
    y_truth = y_test.copy().reset_index(drop=True)
    test_copy = X_test.copy() 
    for i in test_copy.reset_index(drop=True).itertuples():
        if y_pred[i[0]] == 0 and y_truth[i[0]] == 1:
            fps[i[0]] = 1
    test_copy['targetcol'] = np.round(fps, 2)
    return test_copy
    

In [10]:
x_res = label_false_positives(X_test, y_test, y_pred_xgb)
x_res['targetcol'].sum()

326.0

In [11]:
min_sup = 0.1
exp_table_cols = ['Perceived_Race', 'Sex','Age_group__at_arrest_', 'targetcol']
x_res[exp_table_cols].to_csv("temp.csv", index=False)
pd.DataFrame().to_csv("target.csv", index=False)
res = exptbl.calculate_table(f"target.csv", "temp.csv", f"target.csv", min_support_param=min_sup)


Compiling with commands:  ['g++', 'c:\\Users\\Andrew\\Documents\\School\\6320 AI Fairness\\groupproject\\exp\\Explanations.cpp', 'c:\\Users\\Andrew\\Documents\\School\\6320 AI Fairness\\groupproject\\exp\\Lighthouse.cpp', '-o', 'program']
Arguments to the program: target.csv temp.csv 3 15 0 target.csv 0.1
Time: 0:00:08.854436


In [12]:
# read in the new explanation table
exp_table = pd.read_csv(res, sep=";")
exp_table

,Perceived_Race,Sex,Age_group__at_arrest_,targetcol,support
0,*,*,*,0.025,13016
1,0,1,*,0.034,2864
2,7,*,2,0.038,1687
3,*,*,3,0.029,3210
4,7,*,3,0.032,1511
5,*,0,*,0.022,2539
6,*,*,2,0.030,4237
7,*,*,1,0.026,2012
8,*,1,4,0.018,1479
9,*,1,1,0.028,1598


In [13]:
import dice_ml
import recourse_metrics
from raiutils.exceptions import UserConfigValidationException
import json
import ast

In [14]:
def get_test_with_pred_class(X_test, y_pred, num_counterfactuals, target_class, target_column):
    # Concat the predicted values to the data for counterfactual generation
    df_predicted = pd.DataFrame(y_pred, columns=[target_column])
    df_test_pred = pd.concat([X_test.reset_index(drop=True), df_predicted.reset_index(drop=True)], axis=1)
    target_set = df_test_pred.loc[df_test_pred[target_column] == target_class].drop([target_column], axis=1)
    if num_counterfactuals is None:
        num_counterfactuals = target_set.shape[0]
        print(f"setting number of counterfactuals to n={num_counterfactuals}")
    elif num_counterfactuals > target_set.shape[0]:
        num_counterfactuals = target_set.shape[0]
        print(f"warning more counterfactuals requested than num datapoints with label, setting n={num_counterfactuals}")
    target_set = target_set.iloc[0:num_counterfactuals,:].reset_index(drop=True)
    return target_set
# Counterfactual generation func from REACT
def produce_cfs(dataset, cf_method, features_to_perturb, max_cfs_per_sample):
    a = []
    for i in range(dataset.shape[0]):
      sample = dataset.iloc[[i]]
      try:
        cf = cf_method.generate_counterfactuals(sample, 
                                                total_CFs=max_cfs_per_sample, 
                                                desired_class="opposite",
                                                posthoc_sparsity_param=0.1,
                                                posthoc_sparsity_algorithm="linear", 
                                                features_to_vary = features_to_perturb,
                                                verbose=False)
        cfs_list = json.loads(cf.to_json()).get('cfs_list')[0]
        a.append(cfs_list)
      except UserConfigValidationException:
        # no counterfactuals found
        print("no counterfactuals found")
        a.append([])
        continue
    return a
def get_feature_type(feature_name):
    categorical_features = ['Perceived_Race', 'Sex', 'Occurrence_Category', 'ArrestLocDiv']
    ordinal_features = ['Age_group__at_arrest_', 'Arrest_Quarter']
    binary_features = ['IsYouth','Actions_at_arrest___Concealed_i', 'Actions_at_arrest___Combative__',
                  'Actions_at_arrest___Resisted__d', 'Actions_at_arrest___Mental_inst',
                  'Actions_at_arrest___Assaulted_o', 'Actions_at_arrest___Cooperative']
    if feature_name in categorical_features:
        return "categorical"
    elif feature_name in ordinal_features:
        return "ordinal"
    else:
        return "binary"
    
def get_feature_details(df, features_perturb):
    feature_details = {}
    for feature in features_perturb:
            feature_type = get_feature_type(feature)
            
            if feature_type == "numerical":
                min_val, max_val = df[feature].min(), df[feature].max()
                feature_details[feature] = {
                    "type": feature_type,
                    "range_min": min_val,
                    "range_max": max_val
                }

            if feature_type == "ordinal":
                feature_details[feature] = {
                    "type": feature_type,
                    "range_min": df[feature].min(),
                    "range_max": df[feature].max()
                }

            if feature_type == "categorical":
                all_categ_values = sorted(df[feature].unique().tolist())
                feature_details[feature] = {
                    "type": feature_type,
                    "categ_values": all_categ_values
                }
    return feature_details


In [ ]:
# Parameters for counterfacutal testing
max_cfs = 10
max_counterfactuals = 1000#None
target_class = 0 # want to find samples with this classifiaction to find counterfactual of opposite "desired" class
target_column = 'StripSearch'
features_perturb =  ['ArrestLocDiv','Actions_at_arrest___Concealed_i', 'Actions_at_arrest___Combative__',
                  'Actions_at_arrest___Resisted__d', 'Actions_at_arrest___Mental_inst',
                  'Actions_at_arrest___Assaulted_o', 'Actions_at_arrest___Cooperative']
min_sup = 0.1 # Minimum support for explanation tables
explain_features =  ['Perceived_Race', 'Sex', 'Age_group__at_arrest_']


In [21]:
target_set = get_test_with_pred_class(X_test, y_pred_xgb, max_counterfactuals, target_class, target_column)

In [22]:
# Generate Counterfactuals
d = dice_ml.Data(dataframe=pd.concat([X_test, y_test], axis=1), continuous_features = [], outcome_name=target_column)
m = dice_ml.Model(model=xgb_model, backend="sklearn")
exp = dice_ml.Dice(d, m, method="random") 

cfs_out = produce_cfs(target_set, exp, features_perturb, max_cfs)
target_set['cfs_list'] = cfs_out
target_set.to_csv(f"cfs.csv", index=False)

  0%|          | 0/1 [00:00<?, ?it/s]c:\Users\Andrew\anaconda3\envs\REACT\Lib\site-packages\dice_ml\explainer_interfaces\dice_random.py:116: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  candidate_cfs.at[k, selected_features[k][0]] = random_instances.at[k, selected_features[k][0]]
c:\Users\Andrew\anaconda3\envs\REACT\Lib\site-packages\dice_ml\explainer_interfaces\dice_random.py:116: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '7' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  candidate_cfs.at[k, selected_features[k][0]] = random_instances.at[k, selected_features[k][0]]
c:\Users\Andrew\anaconda3\envs\REACT\Lib\site-packages\dice_ml\explainer_interfaces\dice_random.py:116: FutureWarning: Sett

No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec
no counterfactuals found


  0%|          | 0/1 [00:00<?, ?it/s]c:\Users\Andrew\anaconda3\envs\REACT\Lib\site-packages\dice_ml\explainer_interfaces\dice_random.py:116: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  candidate_cfs.at[k, selected_features[k][0]] = random_instances.at[k, selected_features[k][0]]
c:\Users\Andrew\anaconda3\envs\REACT\Lib\site-packages\dice_ml\explainer_interfaces\dice_random.py:116: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  candidate_cfs.at[k, selected_features[k][0]] = random_instances.at[k, selected_features[k][0]]
c:\Users\Andrew\anaconda3\envs\REACT\Lib\site-packages\dice_ml\explainer_interfaces\dice_random.py:116: FutureWarning: Sett

No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec
no counterfactuals found


  0%|          | 0/1 [00:00<?, ?it/s]c:\Users\Andrew\anaconda3\envs\REACT\Lib\site-packages\dice_ml\explainer_interfaces\dice_random.py:116: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  candidate_cfs.at[k, selected_features[k][0]] = random_instances.at[k, selected_features[k][0]]
c:\Users\Andrew\anaconda3\envs\REACT\Lib\site-packages\dice_ml\explainer_interfaces\dice_random.py:116: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  candidate_cfs.at[k, selected_features[k][0]] = random_instances.at[k, selected_features[k][0]]
c:\Users\Andrew\anaconda3\envs\REACT\Lib\site-packages\dice_ml\explainer_interfaces\dice_random.py:116: FutureWarning: Sett

...

  0%|          | 0/1 [00:00<?, ?it/s]c:\Users\Andrew\anaconda3\envs\REACT\Lib\site-packages\dice_ml\explainer_interfaces\dice_random.py:116: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  candidate_cfs.at[k, selected_features[k][0]] = random_instances.at[k, selected_features[k][0]]
c:\Users\Andrew\anaconda3\envs\REACT\Lib\site-packages\dice_ml\explainer_interfaces\dice_random.py:116: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  candidate_cfs.at[k, selected_features[k][0]] = random_instances.at[k, selected_features[k][0]]
c:\Users\Andrew\anaconda3\envs\REACT\Lib\site-packages\dice_ml\explainer_interfaces\dice_random.py:116: FutureWarning: Sett

No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec
no counterfactuals found


  0%|          | 0/1 [00:00<?, ?it/s]c:\Users\Andrew\anaconda3\envs\REACT\Lib\site-packages\dice_ml\explainer_interfaces\dice_random.py:116: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '17' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  candidate_cfs.at[k, selected_features[k][0]] = random_instances.at[k, selected_features[k][0]]
c:\Users\Andrew\anaconda3\envs\REACT\Lib\site-packages\dice_ml\explainer_interfaces\dice_random.py:116: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  candidate_cfs.at[k, selected_features[k][0]] = random_instances.at[k, selected_features[k][0]]
c:\Users\Andrew\anaconda3\envs\REACT\Lib\site-packages\dice_ml\explainer_interfaces\dice_random.py:116: FutureWarning: Set

No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec
no counterfactuals found


  0%|          | 0/1 [00:00<?, ?it/s]c:\Users\Andrew\anaconda3\envs\REACT\Lib\site-packages\dice_ml\explainer_interfaces\dice_random.py:116: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  candidate_cfs.at[k, selected_features[k][0]] = random_instances.at[k, selected_features[k][0]]
c:\Users\Andrew\anaconda3\envs\REACT\Lib\site-packages\dice_ml\explainer_interfaces\dice_random.py:116: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  candidate_cfs.at[k, selected_features[k][0]] = random_instances.at[k, selected_features[k][0]]
c:\Users\Andrew\anaconda3\envs\REACT\Lib\site-packages\dice_ml\explainer_interfaces\dice_random.py:116: FutureWarning: Sett

In [23]:
df = pd.read_csv(f"cfs.csv")
df['cfs_list'] = df['cfs_list'].apply(ast.literal_eval)
df['cfs_list'] = df['cfs_list'].apply(lambda x: np.array(x))
df = df[explain_features + features_perturb + ['cfs_list']]
selected_features = features_perturb
feature_details = get_feature_details(df, selected_features)
r_cost_df = recourse_metrics.recourse_cost_df(df, feature_details, 1)
r_cost_to_summarize = r_cost_df[explain_features+['recourse_cost']]
r_cost_to_summarize.to_csv("temp_cost.csv", index=False)
r_avail_df = recourse_metrics.recourse_availability_df(df)
r_avail_df_to_summarize = r_avail_df[explain_features+['recourse_availability']]
r_avail_df_to_summarize.to_csv("temp_avail.csv", index=False)
r_choice_df = recourse_metrics.recourse_choice_df(df)
r_choice_df_to_summarize = r_choice_df[explain_features+['recourse_choice']]
r_choice_df_to_summarize.to_csv("temp_choice.csv", index=False)

In [24]:
empty_df = pd.DataFrame().to_csv(f"avail_table.csv", index=False) # create empty csv
res_xgb = exptbl.calculate_table(f"avail_table.csv", "temp_avail.csv", f"avail_table.csv", min_support_param=min_sup)
exp_table = pd.read_csv(res_xgb, sep=";")
exp_table

Compiling with commands:  ['g++', 'c:\\Users\\Andrew\\Documents\\School\\6320 AI Fairness\\groupproject\\exp\\Explanations.cpp', 'c:\\Users\\Andrew\\Documents\\School\\6320 AI Fairness\\groupproject\\exp\\Lighthouse.cpp', '-o', 'program']
Arguments to the program: avail_table.csv temp_avail.csv 3 15 0 avail_table.csv 0.1
Time: 0:00:05.395121


,Perceived_Race,Sex,Age_group__at_arrest_,recourse_availability,support
0,*,*,*,0.806,1000
1,*,*,1,0.949,156
2,*,1,2,0.928,265
3,*,*,2,0.926,325
4,*,*,3,0.762,244
5,7,*,2,0.984,127
6,0,1,*,0.864,242
7,7,*,*,0.791,411
8,*,1,3,0.762,189
9,*,*,4,0.681,119


In [25]:
empty_df = pd.DataFrame().to_csv(f"choice_table.csv", index=False) # create empty csv
res_xgb = exptbl.calculate_table(f"choice_table.csv", "temp_choice.csv", f"choice_table.csv", min_support_param=min_sup)
exp_table = pd.read_csv(res_xgb, sep=";")
exp_table

Compiling with commands:  ['g++', 'c:\\Users\\Andrew\\Documents\\School\\6320 AI Fairness\\groupproject\\exp\\Explanations.cpp', 'c:\\Users\\Andrew\\Documents\\School\\6320 AI Fairness\\groupproject\\exp\\Lighthouse.cpp', '-o', 'program']
Arguments to the program: choice_table.csv temp_choice.csv 3 15 0 choice_table.csv 0.1
Time: 0:00:05.323616


,Perceived_Race,Sex,Age_group__at_arrest_,recourse_choice,support
0,*,*,*,7.262,1000
1,*,*,2,8.618,325
2,*,*,1,8.846,156
3,7,*,*,7.426,411
4,*,*,3,6.451,244
5,7,*,2,9.591,127
6,0,1,*,7.579,242
7,*,*,4,6.101,119
8,*,1,2,8.683,265
9,*,1,1,9.016,129


In [26]:
empty_df = pd.DataFrame().to_csv(f"cost_table.csv", index=False) # create empty csv
res_xgb = exptbl.calculate_table(f"cost_table.csv", "temp_cost.csv", f"cost_table.csv", min_support_param=min_sup)
exp_table = pd.read_csv(res_xgb, sep=";")
exp_table

Compiling with commands:  ['g++', 'c:\\Users\\Andrew\\Documents\\School\\6320 AI Fairness\\groupproject\\exp\\Explanations.cpp', 'c:\\Users\\Andrew\\Documents\\School\\6320 AI Fairness\\groupproject\\exp\\Lighthouse.cpp', '-o', 'program']
Arguments to the program: cost_table.csv temp_cost.csv 3 15 0 cost_table.csv 0.1
Time: 0:00:05.299624


,Perceived_Race,Sex,Age_group__at_arrest_,recourse_cost,support
0,*,*,*,0.970,806
1,*,*,2,0.983,301
2,7,1,*,0.972,246
3,*,0,*,0.981,158
4,7,*,*,0.978,325
5,7,*,2,0.992,125
6,*,1,2,0.980,246
7,7,1,2,0.989,90
8,*,*,4,0.975,81
9,*,*,1,0.966,148
